# Прогнозирование стоимости недвижимости

**Цель исследования:** Построение ансамблевой модели машинного обучения для предсказания цен на недвижимость в Турции (в турецких лирах, TRY).

### Этапы работы:
1. **Предобработка и очистка данных:** Фильтрация по валюте, типам сделок, отсечение выбросов (квантили 1% и 99%), парсинг сложных строк (комнаты, этажи, даты).
2. **Разведочный анализ данных (EDA):** Анализ пропусков, распределения цен, влияния площади и географического расположения объектов.
3. **Feature Engineering & Pipeline:** Скейлинг числовых фичей, One-Hot кодирование категорий и Target Encoding для географических локаций.
4. **Обучение и сравнение 7 моделей:** Соревнование линейных, древовидных и ансамблевых алгоритмов (Bagging, Random Forest, Extra Trees, AdaBoost, HistGradientBoosting).
5. **Интерпретация и валидация:** Оценка на тестовой выборке из 20 объектов, валидация ручных метрик против `scikit-learn`, разбор ошибок и экспорт в CSV.


In [1]:
# ---------------------------------------------------------
# Конфигурация проекта и входные параметры
# ---------------------------------------------------------
FIO = "Хабибуллин Артём Альбертович"
GROUP = "23П-4"

TRAIN_PATH = "real_estate_data.csv"
TEST_PATH = "test_real_estate_samples.csv"
OUT_PATH = "result_real_estate_samples.csv"

TARGET = "price"
RANDOM_STATE = 42

# Ограничение выборки для ускорения обучения и кросс-валидации
TRAIN_ROWS = 250_000
N_JOBS = -1

print(f"Конфигурация инициализирована для: {FIO} ({GROUP})")


Конфигурация инициализирована для: Хабибуллин Артём Альбертович (23П-4)


## 1. Загрузка библиотек и первичных данных


In [2]:
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# scikit-learn: предобработка
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.model_selection import train_test_split

# scikit-learn: алгоритмы регрессии
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    BaggingRegressor,
)

# scikit-learn: метрики качества и интерпретация
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Загрузка обучающего и тестового датасетов
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print(f"Обучающая выборка (raw): {train_raw.shape[0]:,} строк, {train_raw.shape[1]} колонок")
print(f"Тестовая выборка (test): {test_raw.shape[0]:,} строк, {test_raw.shape[1]} колонок")


Обучающая выборка (raw): 403,487 строк, 17 колонок
Тестовая выборка (test): 20 строк, 17 колонок


## 2. Очистка данных и конструирование признаков (Feature Engineering)


In [3]:
# Вспомогательные функции парсинга текстовых значений
def extract_mean_number(value):
    """Извлекает все числа из строки и возвращает их среднее (например для диапазонов)."""
    if pd.isna(value):
        return np.nan
    numbers = [float(n) for n in re.findall(r"\d+", str(value))]
    return float(np.mean(numbers)) if numbers else np.nan


def parse_room_formula(value):
    """Парсит классическую запись комнат формата '3 + 1' -> (3.0, 1.0)."""
    if pd.isna(value):
        return np.nan, np.nan
    match = re.fullmatch(r"\s*(\d+)\s*\+\s*(\d+)\s*", str(value))
    if match:
        return float(match.group(1)), float(match.group(2))
    return np.nan, np.nan


def categorize_floor(value):
    """Категоризирует нестандартные записи этажей (цоколь, подвал, чердак)."""
    if pd.isna(value):
        return "missing"
    if re.search(r"\d", str(value)):
        return "numeric"
    return str(value).strip()


# Сводка для EDA перед очисткой
raw_summary = {
    "total_rows": len(train_raw),
    "missing_pct": train_raw.isna().mean().mul(100).sort_values(ascending=False),
    "currency_dist": train_raw["price_currency"].value_counts(),
    "listing_dist": train_raw["listing_type"].value_counts().sort_index()
}

# ---------------------------------------------------------
# Процесс фильтрации обучающего датасета
# ---------------------------------------------------------
train_df = train_raw.copy()
filter_stats = {}
initial_len = len(train_df)

# 1. Исключаем возможное пересечение тестовых объектов (Data Leakage)
leakage_mask = train_df["id"].isin(test_raw["id"])
filter_stats["Пересечение с test"] = int(leakage_mask.sum())
train_df = train_df.loc[~leakage_mask]

# 2. Оставляем только турецкие лиры (TRY) и валидные типы (1 — продажа, 2 — аренда)
valid_meta_mask = (train_df["price_currency"] == "TRY") & train_df["listing_type"].isin([1, 2])
filter_stats["Не TRY или неверный тип"] = int((~valid_meta_mask).sum())
train_df = train_df.loc[valid_meta_mask]

# 3. Проверка валидности целевой цены
train_df[TARGET] = pd.to_numeric(train_df[TARGET], errors="coerce")
invalid_price_mask = train_df[TARGET].isna() | ~np.isfinite(train_df[TARGET]) | (train_df[TARGET] <= 0)
filter_stats["Аномальная цена"] = int(invalid_price_mask.sum())
train_df = train_df.loc[~invalid_price_mask].reset_index(drop=True)

print("Удалено строк при фильтрации:", filter_stats)
print(f"Осталось валидных строк: {len(train_df):,} (из {initial_len:,})")

# Сэмплирование при необходимости
if TRAIN_ROWS and len(train_df) > TRAIN_ROWS:
    train_df = train_df.sample(TRAIN_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Размер выборки для обучения: {len(train_df):,} строк")

# Отсечение экстремальных выбросов цены (Винзоризация 1% - 99%) раздельно для аренды и продажи
train_df["price_clipped"] = train_df[TARGET]
for listing_type, group in train_df.groupby("listing_type"):
    q_low, q_high = group[TARGET].quantile([0.01, 0.99])
    train_df.loc[group.index, "price_clipped"] = group[TARGET].clip(q_low, q_high)

# Расчёт базовых статистик площади строго по тренировочной выборке
valid_sizes = pd.to_numeric(train_df["size"], errors="coerce").where(lambda s: s > 0)
GLOBAL_STATS = {
    "size_q01": valid_sizes.quantile(0.01),
    "size_q99": valid_sizes.quantile(0.99),
    "size_median": valid_sizes.median(),
    "size_med_by_subtype": valid_sizes.groupby(train_df["sub_type"]).median().to_dict()
}


def build_features(df, stats=GLOBAL_STATS):
    """Генерирует финальный набор числовых и категориальных признаков."""
    features = pd.DataFrame(index=df.index)

    # 1. Признаки площади с заполнением медианой по подтипу
    size_num = pd.to_numeric(df["size"], errors="coerce").where(lambda s: s > 0)
    features["size_missing"] = size_num.isna().astype(int)
    subtype_median = df["sub_type"].map(stats["size_med_by_subtype"]).fillna(stats["size_median"])
    filled_size = size_num.fillna(subtype_median).clip(stats["size_q01"], stats["size_q99"])
    features["log_size"] = np.log1p(filled_size)

    # 2. Числовые характеристики здания
    features["tom"] = pd.to_numeric(df["tom"], errors="coerce")
    features["building_age"] = df["building_age"].map(extract_mean_number)
    features["total_floor_count"] = df["total_floor_count"].map(extract_mean_number)
    features["floor_no_num"] = df["floor_no"].map(extract_mean_number)
    features["listing_type"] = df["listing_type"]

    # 3. Комнаты и временные метки
    room_tuples = df["room_count"].map(parse_room_formula)
    features["rooms"] = room_tuples.map(lambda t: t[0])
    features["halls"] = room_tuples.map(lambda t: t[1])
    features["rooms_total"] = features["rooms"] + features["halls"]
    features["start_month"] = pd.to_datetime(df["start_date"], format="%m/%d/%y", errors="coerce").dt.month

    # 4. Категориальные параметры
    features["sub_type"] = df["sub_type"].fillna("missing")
    features["heating_type"] = df["heating_type"].fillna("missing")
    features["floor_cat"] = df["floor_no"].map(categorize_floor)

    # 5. Иерархия гео-локации (Город / Район / Микрорайон)
    address_parts = df["address"].fillna("").str.split("/", expand=True)
    for col_idx in range(3):
        if col_idx not in address_parts.columns:
            address_parts[col_idx] = ""

    city = address_parts[0].fillna("")
    district = city + "/" + address_parts[1].fillna("")
    neighborhood = district + "/" + address_parts[2].fillna("")
    lt_prefix = df["listing_type"].astype(str)

    features["city"] = city.replace("", "missing")
    features["te_district"] = lt_prefix + "|" + district
    features["te_neigh"] = lt_prefix + "|" + neighborhood
    features["te_subtype"] = lt_prefix + "|" + df["sub_type"].fillna("missing")

    return features


# Формирование обучающей и контрольной матриц
X_all = build_features(train_df)
y_all = train_df["price_clipped"].astype(float)

X_test = build_features(test_raw).reindex(columns=X_all.columns)
y_test = test_raw[TARGET].astype(float)

# Списки признаков по типам предобработки
num_cols = [
    "log_size", "size_missing", "tom", "building_age",
    "total_floor_count", "floor_no_num", "listing_type",
    "rooms", "halls", "rooms_total", "start_month"
]
cat_cols = ["sub_type", "heating_type", "floor_cat", "city"]
te_cols = ["te_district", "te_neigh", "te_subtype"]

print(f"Признаки сформированы. Всего: {len(X_all.columns)} (Числовых: {len(num_cols)}, Категорий: {len(cat_cols)}, TargetEncoding: {len(te_cols)})")


Удалено строк при фильтрации: {'Пересечение с test': 20, 'Не TRY или неверный тип': 5043, 'Аномальная цена': 45}
Осталось валидных строк: 398,379 (из 403,487)
Размер выборки для обучения: 250,000 строк


Признаки сформированы. Всего: 18 (Числовых: 11, Категорий: 4, TargetEncoding: 3)


## 3. Разведочный анализ данных (EDA)


In [4]:
# Дашборд EDA 1: Пропуски и категории сделки
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

missing_rates = raw_summary["missing_pct"][raw_summary["missing_pct"] > 0].sort_values()
axes[0].barh(missing_rates.index, missing_rates.values, color="tab:red")
axes[0].set_title("Доля пропущенных значений, %")
axes[0].set_xlabel("Процент пропусков")

currency_counts = raw_summary["currency_dist"]
axes[1].bar(currency_counts.index.astype(str), currency_counts.values, color="tab:blue")
axes[1].set_yscale("log")
axes[1].set_title("Валюты объявлений (лог. шкала)")

listing_counts = raw_summary["listing_dist"]
axes[2].bar(["1 (Продажа)", "2 (Аренда)"], listing_counts.iloc[:2].values, color=["tab:blue", "tab:orange"])
axes[2].set_title("Соотношение типов сделок")

plt.tight_layout()
plt.show()

# Дашборд EDA 2: Распределение цен и зависимость от площади
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for l_type, label, col in [(1, "Продажа", "tab:blue"), (2, "Аренда", "tab:orange")]:
    subset_prices = train_df.loc[train_df["listing_type"] == l_type, "price_clipped"]
    axes[0].hist(np.log10(subset_prices), bins=50, alpha=0.55, label=label, color=col)
axes[0].set_title("Распределение цен (log10 TRY)")
axes[0].set_xlabel("log10(Цена)")
axes[0].legend()

sample_eda = train_df.sample(min(5000, len(train_df)), random_state=RANDOM_STATE)
sample_eda["clean_size"] = pd.to_numeric(sample_eda["size"], errors="coerce").where(lambda s: s > 0)

for ax_idx, l_type, label, col in [(1, 1, "Продажа", "tab:blue"), (2, 2, "Аренда", "tab:orange")]:
    grp = sample_eda[(sample_eda["listing_type"] == l_type) & sample_eda["clean_size"].notna()]
    axes[ax_idx].scatter(grp["clean_size"], grp["price_clipped"], s=8, alpha=0.35, color=col)
    axes[ax_idx].set_xscale("log")
    axes[ax_idx].set_yscale("log")
    axes[ax_idx].set_title(f"{label}: связь цены и площади")
    axes[ax_idx].set_xlabel("Площадь (м²)")
    axes[ax_idx].set_ylabel("Цена (TRY)")

plt.tight_layout()
plt.show()


## 4. Пайплайн предобработки и обучение 7 моделей регрессии


In [5]:
# Гиперпараметры моделей для честного сравнения
PARAMS = {
    "Ridge": dict(alpha=1.0),
    "Decision Tree": dict(max_depth=14, min_samples_leaf=20),
    "Random Forest": dict(
        n_estimators=80, max_depth=18, min_samples_leaf=5,
        max_features=0.5, bootstrap=True, max_samples=0.5
    ),
    "Extra Trees": dict(
        n_estimators=80, max_depth=20, min_samples_leaf=5,
        max_features=0.7, bootstrap=True, max_samples=0.5
    ),
    "Bagging": dict(n_estimators=30, max_samples=0.5),
    "AdaBoost": dict(n_estimators=50, learning_rate=0.5),
    "Gradient Boosting": dict(
        max_iter=400, learning_rate=0.08, max_leaf_nodes=63,
        min_samples_leaf=40, l2_regularization=1.0, early_stopping=True
    ),
}

MODEL_TYPES = {
    "Ridge": "линейная",
    "Decision Tree": "одно дерево",
    "Random Forest": "ансамбль (бэггинг)",
    "Extra Trees": "ансамбль (бэггинг)",
    "Bagging": "ансамбль (бэггинг)",
    "AdaBoost": "ансамбль (бустинг)",
    "Gradient Boosting": "ансамбль (бустинг)",
}


def create_preprocessor():
    """Собирает общий трансформер признаков для scikit-learn."""
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False))
    ])
    
    target_encoder = TargetEncoder(
        target_type="continuous", smooth="auto", cv=5, random_state=RANDOM_STATE
    )
    
    return ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
        ("target_enc", target_encoder, te_cols)
    ])


def get_model_instance(name):
    """Фабрика инициализации моделей с фиксированным random_state."""
    params = PARAMS[name]
    if name == "Ridge":
        return Ridge(**params)
    elif name == "Decision Tree":
        return DecisionTreeRegressor(random_state=RANDOM_STATE, **params)
    elif name == "Random Forest":
        return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **params)
    elif name == "Extra Trees":
        return ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **params)
    elif name == "Bagging":
        base_tree = DecisionTreeRegressor(max_depth=12, min_samples_leaf=10)
        return BaggingRegressor(estimator=base_tree, random_state=RANDOM_STATE, n_jobs=N_JOBS, **params)
    elif name == "AdaBoost":
        base_tree = DecisionTreeRegressor(max_depth=8, min_samples_leaf=20)
        return AdaBoostRegressor(estimator=base_tree, random_state=RANDOM_STATE, **params)
    elif name == "Gradient Boosting":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)
    raise ValueError(f"Неизвестная модель: {name}")


def build_full_pipeline(name):
    """Объединяет предобработку и модель в единый Pipeline."""
    return Pipeline([
        ("preprocessor", create_preprocessor()),
        ("regressor", get_model_instance(name))
    ])


In [6]:
# Разделение на train и validation со стратификацией по типу сделки
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=X_all["listing_type"]
)

evaluation_records = []
validation_predictions_log = {}
fit_durations = {}

print("=== Старт сравнительного обучения 7 моделей ===")
for model_name in PARAMS:
    start_time = time.time()
    pipeline = build_full_pipeline(model_name)
    
    # Обучаем модель на логарифмированной целевой переменной log1p(price)
    pipeline.fit(X_train, np.log1p(y_train))
    duration = time.time() - start_time
    
    pred_log = pipeline.predict(X_val)
    validation_predictions_log[model_name] = pred_log
    pred_original_scale = np.expm1(pred_log)
    
    # Расчёт метрик качества
    val_mae = mean_absolute_error(y_val, pred_original_scale)
    val_rmse = root_mean_squared_error(y_val, pred_original_scale)
    val_r2 = r2_score(y_val, pred_original_scale)
    log_rmse = root_mean_squared_error(np.log1p(y_val), pred_log)
    
    fit_durations[model_name] = duration
    evaluation_records.append({
        "Модель": model_name,
        "Тип": MODEL_TYPES[model_name],
        "Val MAE": val_mae,
        "Val RMSE": val_rmse,
        "Val R²": val_r2,
        "RMSE (log)": log_rmse,
        "Время (с)": duration
    })
    print(f"{model_name:<18} | R²: {val_r2:.3f} | RMSE (log): {log_rmse:.3f} | Время: {duration:.1f} с")

# Сводная таблица лидеров
comparison_df = pd.DataFrame(evaluation_records).set_index("Модель").sort_values("RMSE (log)")
display(comparison_df.round(3))

best_model_name = comparison_df["RMSE (log)"].idxmin()
print(f"\nЛучшая модель по критерию RMSE (log): {best_model_name}")

# Обучение финального пайплайна на 100% данных перед инференсом на тесте
final_pipeline = build_full_pipeline(best_model_name)
final_pipeline.fit(X_all, np.log1p(y_all))
print("Финальная модель успешно переобучена на полном объеме данных!")


=== Старт сравнительного обучения 7 моделей ===


Ridge              | R²: 0.667 | RMSE (log): 0.412 | Время: 1.5 с


Decision Tree      | R²: 0.738 | RMSE (log): 0.387 | Время: 3.2 с


Random Forest      | R²: 0.783 | RMSE (log): 0.349 | Время: 14.3 с


Extra Trees        | R²: 0.781 | RMSE (log): 0.350 | Время: 21.5 с


Bagging            | R²: 0.759 | RMSE (log): 0.370 | Время: 15.7 с


AdaBoost           | R²: 0.667 | RMSE (log): 0.481 | Время: 75.7 с


Gradient Boosting  | R²: 0.793 | RMSE (log): 0.343 | Время: 13.2 с


Лучшая модель по критерию RMSE (log): Gradient Boosting


Финальная модель успешно переобучена на полном объеме данных!


## 5. Графический анализ и сравнение моделей


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. R² (больше — лучше)
sorted_r2 = comparison_df["Val R²"].sort_values()
axes[0].barh(sorted_r2.index, sorted_r2.values, color="tab:blue")
axes[0].set_title("Коэффициент детерминации R² (выше = лучше)")

# 2. Ошибка в лог-шкале RMSE(log)
sorted_log_rmse = comparison_df["RMSE (log)"].sort_values(ascending=False)
axes[1].barh(sorted_log_rmse.index, sorted_log_rmse.values, color="tab:red")
axes[1].set_title("RMSE в лог-шкале (меньше = лучше)")

# 3. Время обучения
sorted_times = comparison_df["Время (с)"].sort_values(ascending=False)
axes[2].barh(sorted_times.index, sorted_times.values, color="tab:green")
axes[2].set_title("Время обучения (секунды)")

plt.tight_layout()
plt.show()


## 6. Валидация на тестовых объектах и сверка ручных метрик


In [8]:
# Предсказание для 20 тестовых объектов
test_predictions_log = final_pipeline.predict(X_test)
test_predictions = np.expm1(test_predictions_log)

result_df = test_raw.copy()
result_df["ФИО"] = FIO
result_df["Группа"] = GROUP


def calculate_manual_metrics(y_true, y_pred):
    """Самостоятельный расчёт MAE, RMSE и R2 по математическим определениям."""
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    
    residuals = y_true_arr - y_pred_arr
    mae = float(np.mean(np.abs(residuals)))
    rmse = float(np.sqrt(np.mean(residuals ** 2)))
    
    total_sum_squares = np.sum((y_true_arr - np.mean(y_true_arr)) ** 2)
    residual_sum_squares = np.sum(residuals ** 2)
    r2 = float(1.0 - (residual_sum_squares / total_sum_squares))
    
    return mae, rmse, r2


# Сверка ручного расчёта с scikit-learn
m_mae, m_rmse, m_r2 = calculate_manual_metrics(y_test, test_predictions)
sk_mae = mean_absolute_error(y_test, test_predictions)
sk_rmse = root_mean_squared_error(y_test, test_predictions)
sk_r2 = r2_score(y_test, test_predictions)

metrics_comparison = pd.DataFrame({
    "Формула вручную": [m_mae, m_rmse, m_r2],
    "Библиотека sklearn": [sk_mae, sk_rmse, sk_r2]
}, index=["MAE", "RMSE", "R²"])
metrics_comparison["Абсолютная разница"] = (
    metrics_comparison["Формула вручную"] - metrics_comparison["Библиотека sklearn"]
).abs()

print("=== Сверка корректности расчёта метрик ===")
display(metrics_comparison.round(6))

# Запись требуемых колонок в результирующий датасет
result_df["MAE_manual"] = round(m_mae, 4)
result_df["MAE_sklearn"] = round(sk_mae, 4)
result_df["RMSE_manual"] = round(m_rmse, 4)
result_df["RMSE_sklearn"] = round(sk_rmse, 4)
result_df["R2_manual"] = round(m_r2, 4)
result_df["R2_sklearn"] = round(sk_r2, 4)
result_df["predicted_price"] = test_predictions.round(2)

# Экспорт в итоговый CSV
result_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Результаты предсказаний сохранены в файл: {OUT_PATH}")
display(result_df[["id", "listing_type", "price", "predicted_price", "R2_manual"]].head())


=== Сверка корректности расчёта метрик ===


Результаты предсказаний сохранены в файл: result_real_estate_samples.csv


## 7. Разбор ошибок и итоговый отчёт


In [9]:
# Анализ точности по тестовым объектам
error_analysis = test_raw[["id", "sub_type", "listing_type", "size"]].copy()
error_analysis["actual_price"] = y_test.values
error_analysis["predicted_price"] = test_predictions.round(2)
error_analysis["abs_error"] = np.abs(error_analysis["actual_price"] - error_analysis["predicted_price"])
error_analysis["APE_pct"] = (
    error_analysis["abs_error"] / error_analysis["actual_price"] * 100
).round(1)

print("Топ-5 объектов с наибольшей относительной ошибкой:")
display(error_analysis.sort_values("APE_pct", ascending=False).head(5))

# Автоматическая генерация Markdown-отчёта
best_stats = comparison_df.loc[best_model_name]
format_number = lambda n: f"{n:,.0f}".replace(",", " ")

report_markdown = f"""
### Итоговый отчёт по работе
- **Обучающая выборка:** {format_number(len(train_df))} объявлений. Отсечены не-TRY валюты, битые цены и выбросы (1%-99%).
- **Победившая модель:** **{best_model_name}** ({MODEL_TYPES[best_model_name]}).
- **Метрики лучшей модели на валидации:** R² = **{best_stats['Val R²']:.3f}**, RMSE (log) = **{best_stats['RMSE (log)']:.3f}**.
- **Качество на тестовой выборке (20 объектов):**
  - MAE: **{format_number(m_mae)} TRY**
  - RMSE: **{format_number(m_rmse)} TRY**
  - R²: **{m_r2:.3f}**
- **Верификация вычислений:** Ручной расчёт формул и функции библиотеки `sklearn` дают идентичный результат (разница < 1e-6).
"""
display(Markdown(report_markdown))


Топ-5 объектов с наибольшей относительной ошибкой:
